In [1]:
import os
from src.arguments import ModelArguments, DataArguments
from src.model.model import MMEBModel
from src.model.processor import load_processor, QWEN2_VL, VLM_IMAGE_TOKENS, \
    Qwen2_VL_process_fn, LLAVA_QWEN2, FastVLM_process_fn
from src.utils import batch_to_device
from PIL import Image
import numpy as np
from src.model.llava.model import LlavaQwen2ForCausalLM
import torch
import math
%matplotlib inline
import matplotlib.pyplot as plt
import torch.nn.functional as F

from transformers.image_transforms import (
    convert_to_rgb,
    resize,
)

/home/hungpv/projects/Talas_VLM_Embed/vlm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/mnt/hungpv/projects/Talas_VLM_Embed/src/model/vlm_backbone/internvideo2/modeling_internvideo2.py:541: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)
[2026-08-12 18:41:33,096] DEBUG [matplotlib:347] matplotlib data path: /home/hungpv/projects/Talas_VLM_Embed/vlm/lib/python3.11/site-packages/matplotlib/mpl-data
[2026-08-12 18:41:33,102] DEBUG [matplotlib:347] CONFIGDIR=/mnt/hungpv/.config/matplotlib
[2026-08-12 18:41:33,110] DEBUG [matplotlib:1564] interactive is False
[2026-08-12 18:41:33,111] DEBUG [matplotlib:1565] platform is linux
[2026-08-12 18:41:33,132] DEBUG [matplotlib:347] CACHEDIR=/mnt/hu

FusedMLP of flash_attn is not installed!!!
DropoutAddRMSNorm of flash_attn is not installed!!!
flash_attn_interface or bert_padding of flash_attn is not installed!!!


[2026-08-12 18:41:33,347] DEBUG [matplotlib.pyplot:517] Loaded backend module://matplotlib_inline.backend_inline version unknown.
[2026-08-12 18:41:33,350] DEBUG [matplotlib.pyplot:517] Loaded backend inline version unknown.


In [2]:
model_args = ModelArguments(
    model_name='raghavlite/B3_Qwen2_2B',
    pooling='last',
    normalize=True,
    model_backbone='qwen2_vl',
    lora=True
)
process_fn = Qwen2_VL_process_fn
token_img = QWEN2_VL

# model_args = ModelArguments(
#     model_name='apple/FastVLM-0.5B',
#     pooling='last',
#     normalize=True,
#     model_backbone=LLAVA_QWEN2,
#     lora=True,
# )
# process_fn = FastVLM_process_fn
# token_img = LLAVA_QWEN2

data_args = DataArguments()

processor = load_processor(model_args, None)
model = MMEBModel.build(model_args)
model = model.to('cuda', dtype=torch.bfloat16)
model.eval()

[2026-08-12 18:41:33,362] INFO [src.utils:21] Loading processor from: raghavlite/B3_Qwen2_2B
[2026-08-12 18:41:33,368] DEBUG [urllib3.connectionpool:1062] Starting new HTTPS connection (1): huggingface.co:443


Load Qwen2-VL processor
>>>>>>>>>>>>>>>>>>>>>>>> Processor raghavlite/B3_Qwen2_2B


[2026-08-12 18:41:33,625] DEBUG [urllib3.connectionpool:544] https://huggingface.co:443 "GET /api/models/raghavlite/B3_Qwen2_2B/revision/main HTTP/1.1" 200 None
Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 693.62it/s]
[2026-08-12 18:41:33,882] DEBUG [urllib3.connectionpool:544] https://huggingface.co:443 "HEAD /raghavlite/B3_Qwen2_2B/resolve/main/tokenizer_config.json HTTP/1.1" 307 0
[2026-08-12 18:41:33,892] DEBUG [urllib3.connectionpool:544] https://huggingface.co:443 "HEAD /api/resolve-cache/models/raghavlite/B3_Qwen2_2B/9dced36897b37e4b31863617f3837c9d664e1792/tokenizer_config.json HTTP/1.1" 200 0
[2026-08-12 18:41:34,124] DEBUG [urllib3.connectionpool:544] https://huggingface.co:443 "GET /api/models/raghavlite/B3_Qwen2_2B/tree/main/additional_chat_templates?recursive=False&expand=False HTTP/1.1" 404 64
[2026-08-12 18:41:34,726] DEBUG [urllib3.connectionpool:544] https://huggingface.co:443 "GET /api/models/raghavlite/B3_Qwen2_2B/revision/main HTTP/1.1" 200 None
Fetching 1 f

ImageProcessor type: <class 'transformers.models.qwen2_vl.image_processing_qwen2_vl.Qwen2VLImageProcessor'>
teacher processor loaded here.


[2026-08-12 18:41:38,519] DEBUG [urllib3.connectionpool:544] https://huggingface.co:443 "HEAD /raghavlite/B3_Qwen2_2B/resolve/main/config.json HTTP/1.1" 307 0
[2026-08-12 18:41:38,529] DEBUG [urllib3.connectionpool:544] https://huggingface.co:443 "HEAD /api/resolve-cache/models/raghavlite/B3_Qwen2_2B/9dced36897b37e4b31863617f3837c9d664e1792/config.json HTTP/1.1" 200 0
[2026-08-12 18:41:38,535] INFO [src.utils:21] Loading backbone [qwen2_vl] from raghavlite/B3_Qwen2_2B
`torch_dtype` is deprecated! Use `dtype` instead!


Detected model type: qwen2_vl
Determined model backbone: qwen2_vl


[2026-08-12 18:41:38,769] DEBUG [urllib3.connectionpool:544] https://huggingface.co:443 "HEAD /Qwen/Qwen2-VL-2B-Instruct/resolve/main/model.safetensors HTTP/1.1" 404 0
[2026-08-12 18:41:39,005] DEBUG [urllib3.connectionpool:544] https://huggingface.co:443 "HEAD /Qwen/Qwen2-VL-2B-Instruct/resolve/main/model.safetensors.index.json HTTP/1.1" 307 0
[2026-08-12 18:41:39,013] DEBUG [urllib3.connectionpool:544] https://huggingface.co:443 "HEAD /api/resolve-cache/models/Qwen/Qwen2-VL-2B-Instruct/895c3a49bc3fa70a340399125c650a463535e71c/model.safetensors.index.json HTTP/1.1" 200 0
[2026-08-12 18:41:39,250] DEBUG [urllib3.connectionpool:544] https://huggingface.co:443 "GET /api/models/Qwen/Qwen2-VL-2B-Instruct/revision/main HTTP/1.1" 200 None
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 25.88it/s]
[2026-08-12 18:41:39,641] DEBUG [urllib3.connectionpool:544] https://huggingface.co:443 "HEAD /Qwen/Qwen2-VL-2B-Instruct/resolve/main/generation_config.json HTTP/1.1" 307 0
[2026-08-12

MMEBModel(
  (encoder): PeftModel(
    (base_model): LoraModel(
      (model): Qwen2VLForConditionalGeneration(
        (visual): Qwen2VisionTransformerPretrainedModel(
          (patch_embed): PatchEmbed(
            (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
          )
          (rotary_pos_emb): VisionRotaryEmbedding()
          (blocks): ModuleList(
            (0-31): 32 x Qwen2VLVisionBlock(
              (norm1): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
              (norm2): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
              (attn): VisionAttention(
                (qkv): Linear(in_features=1280, out_features=3840, bias=True)
                (proj): Linear(in_features=1280, out_features=1280, bias=True)
              )
              (mlp): VisionMlp(
                (fc1): Linear(in_features=1280, out_features=5120, bias=True)
                (act): QuickGELUActivation()
                (fc2): Linear(in_feat

In [6]:
processor_inputs = {
    "text": [f'{VLM_IMAGE_TOKENS[token_img]} Represent the given image with the following question: What is in the image',
          f'{VLM_IMAGE_TOKENS[token_img]} Represent the given image with the following question: What is in the image'],
    "images": [Image.open('example.jpg').resize((500, 1025)),
            Image.open('example.jpg')],
}

inputs = process_fn(
    processor_inputs,
    processor, 
    # square_padding=True
    )
inputs = batch_to_device(inputs, "cuda")
inputs['images'][0], inputs['images'][1]



(<PIL.Image.Image image mode=RGB size=500x1025>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=375x500>)

In [7]:
input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]

eos_id = processor.tokenizer.eos_token_id
last_idx = attention_mask.long().sum(dim=1) - 1
last_ids = input_ids[torch.arange(input_ids.size(0)), last_idx]

print("eos_id:", eos_id)
print("last_ids:", last_ids.tolist())
print("all end with eos:", bool((last_ids == eos_id).all()))
print("all special ids: ", processor.tokenizer.all_special_ids)

eos_id: 151645
last_ids: [2168, 151643]
all end with eos: False
all special ids:  [151645, 151643, 151644, 151646, 151647, 151648, 151649, 151650, 151651, 151652, 151653, 151654, 151655, 151656]


In [ ]:
type(model.encoder.get_vision_tower().vision_tower.model)

In [ ]:
output = model.encode_input(inputs)
output

In [ ]:
x = torch.randn(1, 3, 768, 768).to('cuda', dtype=torch.bfloat16)
with torch.no_grad():
    y = model.encoder.get_vision_tower()(x)
y.shape